# NB06: Cross-Validation with Pangenome & Literature

Validate cultivation-bias signals by comparing against:
1. eggNOG/GapMind annotations for 147 ENIGMA genomes linked to `kbase_ke_pangenome`
2. Published ORFRC metagenome studies

**Requires**: NB04 outputs, BERDL Spark access

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from berdl_notebook_utils.setup_spark_session import get_spark_session

spark = get_spark_session()
DATA_DIR = Path('../data')
FIG_DIR = Path('../figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 1. Identify Pangenome-Linked ENIGMA Genomes

In [ ]:
# 147 ENIGMA genomes match kbase_ke_pangenome via GCF/GCA accessions
# Previously identified: 108 GCF + 39 GCA matches

cult_meta = pd.read_csv(DATA_DIR / 'cultured_genome_metadata.tsv', sep='\t')
linked = cult_meta[cult_meta['external_id'].notna() & 
                   cult_meta['external_id'].str.startswith(('GCF_', 'GCA_'))]
print(f'{len(linked)} ENIGMA genomes with GCF/GCA accessions')

# Match to pangenome
accessions = linked['external_id'].tolist()
acc_str = ','.join([f"'{a}'" for a in accessions])

pangenome_match = spark.sql(f'''
    SELECT genome_id, genome_name, cluster_id
    FROM kbase_ke_pangenome.genome
    WHERE genome_name IN ({acc_str})
    OR genome_id IN ({acc_str})
''').toPandas()

print(f'{len(pangenome_match)} matches in kbase_ke_pangenome')

## 2. eggNOG Annotations for Linked Genomes

In [ ]:
if len(pangenome_match) > 0:
    cluster_ids = pangenome_match['cluster_id'].unique()
    cid_str = ','.join([f"'{c}'" for c in cluster_ids[:50]])  # batch
    
    eggnog = spark.sql(f'''
        SELECT 
            cluster_id, kegg_ko, cog_category, description
        FROM kbase_ke_pangenome.eggnog_mapper_annotations
        WHERE cluster_id IN ({cid_str})
        AND kegg_ko IS NOT NULL AND kegg_ko != ''
    ''').toPandas()
    
    print(f'{len(eggnog)} eggNOG annotations with KO assignments')
    print(f'{eggnog["kegg_ko"].nunique()} unique KOs from eggNOG')
    
    # Compare with our genome depot KO annotations
    cult_ko = pd.read_csv(DATA_DIR / 'cultured_ko_profiles.tsv', sep='\t')
    depot_kos = set(cult_ko['ko_id'].unique())
    eggnog_kos = set(eggnog['kegg_ko'].unique())
    
    overlap = depot_kos & eggnog_kos
    print(f'\nKO overlap: {len(overlap)} shared')
    print(f'  Depot-only: {len(depot_kos - eggnog_kos)}')
    print(f'  eggNOG-only: {len(eggnog_kos - depot_kos)}')
else:
    print('No pangenome-linked genomes found')

## 3. Annotation Concordance

In [ ]:
# For the 147 linked genomes, compare KO sets from two sources:
# 1. Genome Depot (browser_protein_kegg_orthologs)
# 2. eggNOG (via pangenome)
# High concordance = our cultured KO profiles are reliable

if len(pangenome_match) > 0 and len(eggnog) > 0:
    linked_genome_ids = linked['genome_id'].tolist()
    depot_linked_kos = set(cult_ko[cult_ko['genome_id'].isin(linked_genome_ids)]['ko_id'].unique())
    eggnog_linked_kos = set(eggnog['kegg_ko'].unique())
    
    concordance = len(depot_linked_kos & eggnog_linked_kos) / len(depot_linked_kos | eggnog_linked_kos)
    print(f'Jaccard concordance (linked genomes only): {concordance:.3f}')
    print(f'  Depot KOs: {len(depot_linked_kos)}')
    print(f'  eggNOG KOs: {len(eggnog_linked_kos)}')
    print(f'  Intersection: {len(depot_linked_kos & eggnog_linked_kos)}')

## 4. Literature Comparison

Compare our cultivation-bias signals against published ORFRC studies.

In [ ]:
# Load cultivation coverage results
coverage = pd.read_csv(DATA_DIR / 'ko_cultivation_coverage_full.tsv', sep='\t', index_col=0)
marker_cov = pd.read_csv(DATA_DIR / 'marker_cultivation_coverage.tsv', sep='\t')

# Literature predictions vs our findings
predictions = [
    ('Wood-Ljungdahl', 'depleted', 'Beaver & Neufeld 2024'),
    ('NiFe-hydrogenase', 'depleted', 'Bagnoud 2016'),
    ('Sulfate reduction', 'enriched', 'clay_confined_subsurface'),
    ('Methanogenesis', 'depleted', 'Beaver & Neufeld 2024'),
    ('Denitrification', 'variable', 'Wu 2023'),
    ('N-fixation', 'variable', 'Beaver & Neufeld 2024'),
    ('Metal resistance', 'variable', 'Goff 2024'),
    ('Motility/chemotaxis', 'enriched', 'Cultivation selection'),
    ('Aerobic respiration', 'enriched', 'Cultivation selection'),
    ('Conjugation/MGE', 'depleted', 'Kothari 2019'),
]

print(f'{"Category":<25} {"Predicted":>10} {"Observed log2R":>15} {"Match?":>8} {"Source"}')
print('-' * 80)
for cat, pred, source in predictions:
    sub = marker_cov[marker_cov['category'] == cat]
    if len(sub) > 0:
        mean_lr = sub['log2_ratio'].mean()
        if pred == 'enriched':
            match = 'YES' if mean_lr > 0 else 'NO'
        elif pred == 'depleted':
            match = 'YES' if mean_lr < 0 else 'NO'
        else:
            match = '~'
        print(f'{cat:<25} {pred:>10} {mean_lr:>+15.2f} {match:>8} {source}')
    else:
        print(f'{cat:<25} {pred:>10} {"N/A":>15} {"N/A":>8} {source}')